In [61]:
# ════════════════════════════════════════════════════════════════
# CELL 1 ▸ Imports & Constants
# ════════════════════════════════════════════════════════════════
import os
import io
import time
import shutil
import glob
import pathlib
import json
import requests
import pandas as pd
import polars as pl
from datetime import datetime, timedelta

from PIL import Image
import win32clipboard

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import TimeoutException, NoAlertPresentException
from urllib.parse import urlparse

# ── Webhook (same as lc_2_capture) ──────────────────────────────
TEAMS_WEBHOOK_URL = "https://default599e51d62f8c43478e591f795a51a9.8c.environment.api.powerplatform.com:443/powerautomate/automations/direct/workflows/c24f30c010df45a6a6dac9421643bb34/triggers/manual/paths/invoke?api-version=1&sp=%2Ftriggers%2Fmanual%2Frun&sv=1.0&sig=5vWDl18a7-IWSvHuZAWgGtQcwM54nEapSArj4JVPnGg"


In [62]:
# ════════════════════════════════════════════════════════════════
# CELL 2 ▸ Browser Setup · Login · Download CSV
# ════════════════════════════════════════════════════════════════
expedia_url   = "https://console.vap.expedia.com/analytics-console-user-interface/optics/agentRealtime"
report_name   = "Realtime Outage"
wait_seconds  = 10
start_time    = datetime.now()

parsed_url    = urlparse(expedia_url)
path_fragment = parsed_url.path.split('/')[-1]

chrome_options = Options()
chrome_options.add_argument(r'--user-data-dir=C:/temp/new_chrome_profile')
chrome_options.add_argument(r'--profile-directory=Default')
chrome_options.add_argument("--start-maximized")

service = Service(
    r"C:\Users\huuchinh.nguyen\Concentrix Corporation"
    r"\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE"
    r"\chromedriver-win64\chromedriver.exe"
)
driver = webdriver.Chrome(service=service, options=chrome_options)


def send_to_clipboard(image):
    output = io.BytesIO()
    image.convert("RGB").save(output, "BMP")
    data = output.getvalue()[14:]
    output.close()
    win32clipboard.OpenClipboard()
    win32clipboard.EmptyClipboard()
    win32clipboard.SetClipboardData(win32clipboard.CF_DIB, data)
    win32clipboard.CloseClipboard()


def check_and_login(driver, expedia_url, wait_time=10):
    driver.get(expedia_url)
    time.sleep(10)
    try:
        sign_in_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[data-testid="console-okta-sign-in"]'))
        )
        print("\ud83d\udd11 Sign-in required! Clicking...")
        sign_in_button.click()
        time.sleep(2)
        try:
            keep_signed_in_label = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, 'label[for="input36"][data-se-for-name="rememberMe"]')
                )
            )
            keep_signed_in_label.click()
            time.sleep(1)
        except TimeoutException:
            print("No 'Keep me signed in' option. Skipping.")
        try:
            next_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, 'input.button.button-primary[type="submit"][value="Next"]')
                )
            )
            next_button.click()
            time.sleep(10)
        except TimeoutException:
            print("No 'Next' button. Skipping.")
        print("\ud83c\udf89 Login successful!")
        try:
            driver.switch_to.alert.accept()
        except NoAlertPresentException:
            pass
        driver.get(expedia_url)
    except TimeoutException:
        print("\u2705 No sign-in required. Continuing...")


check_and_login(driver, expedia_url)
wait = WebDriverWait(driver, 10)

# ── Download 'Logged-In Agents' CSV (same JS title-search as lc_2_capture) ──
try:
    target_btn = wait.until(lambda d: d.execute_script("""
        const titleEl = Array.from(document.querySelectorAll('*')).find(el =>
            el.childNodes.length === 1 &&
            el.childNodes[0].nodeType === Node.TEXT_NODE &&
            el.textContent.trim() === 'Logged-In Agents'
        );
        if (!titleEl) return null;
        let node = titleEl.parentElement;
        while (node && node !== document.body) {
            const btns = node.querySelectorAll('button.settingsButton');
            if (btns.length === 1) return btns[0];
            node = node.parentElement;
        }
        return null;
    """))

    if target_btn is None:
        raise Exception("Kh\u00f4ng t\u00ecm th\u1ea5y settingsButton c\u1ee7a 'Logged-In Agents'")

    print("\u2705 Found settingsButton (Logged-In Agents)")
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", target_btn)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", target_btn)
    print("\u2705 Clicked settingsButton")

    wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "div.uitk-menu-container[aria-hidden='false']")
    ))

    download_csv_button = wait.until(
        EC.element_to_be_clickable((
            By.XPATH,
            "//div[contains(@class,'uitk-menu-open')][@aria-hidden='false']"
            "//span[text()='Download CSV']/ancestor::button"
        ))
    )
    download_csv_button.click()
    print("\u2705 Clicked Download CSV")

except Exception as e:
    print(f"\u274c L\u1ed7i: {e}")

wait.until(EC.url_contains(path_fragment))
time.sleep(wait_seconds)

driver.quit()


✅ No sign-in required. Continuing...
✅ Found settingsButton (Logged-In Agents)
✅ Clicked settingsButton
✅ Clicked Download CSV


In [63]:
# ════════════════════════════════════════════════════════════════
# CELL 3 ▸ Move File → current_agent
# ════════════════════════════════════════════════════════════════
source_folder = r"C:\temp\expedia_downloads"
destination_folder = (
    r"C:\Users\huuchinh.nguyen\Concentrix Corporation"
    r"\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\current_agent"
)

file_patterns = [
    os.path.join(source_folder, "Logged-In Agents*.csv"),
    os.path.join(source_folder, "Logged-In Agents*.xlsx"),
]

for pattern in file_patterns:
    for filepath in glob.glob(pattern):
        filename = os.path.basename(filepath)
        destination_path = os.path.join(destination_folder, filename)
        shutil.move(filepath, destination_path)
        print(f"Moved: {filepath} -> {destination_path}")


Moved: C:\temp\expedia_downloads\Logged-In Agents-Thu Jul 30 2026 09_22_13 GMT+0700 (Indochina Time).csv -> C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\current_agent\Logged-In Agents-Thu Jul 30 2026 09_22_13 GMT+0700 (Indochina Time).csv


In [64]:
# ════════════════════════════════════════════════════════════════
# CELL 4 ▸ Load · Transform · Build Tables
# ════════════════════════════════════════════════════════════════
DATA_DIR = (
    r"C:\Users\huuchinh.nguyen\Concentrix Corporation"
    r"\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\current_agent"
)

def convert_to_datetime(struct_time):
    return datetime(*struct_time[:6])

def input_data(data_dir):
    list_files = []
    for filename in pathlib.Path(data_dir).glob('**/*.*'):
        file_suffixes = filename.suffixes
        if not (file_suffixes and file_suffixes[-1].lower() in ['.xlsx', '.csv']):
            continue
        export_time          = os.path.getmtime(filename)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))
        file_name            = filename.stem
        try:
            if file_suffixes[-1].lower() == '.xlsx':
                df = pl.read_excel(filename)
                if df.is_empty():
                    print(f"⚠️  Empty file skipped: {filename.name}")
                    continue
            elif file_suffixes[-1].lower() == '.csv':
                if os.path.getsize(filename) == 0:
                    print(f"⚠️  Zero-size file skipped: {filename.name}")
                    continue
                df = pl.read_csv(filename, infer_schema_length=10000)
                if df.is_empty():
                    print(f"⚠️  Empty file skipped: {filename.name}")
                    continue
            df = df.with_columns([
                pl.lit(file_name).alias('sheet_name'),
                pl.lit(export_time_datetime).alias('Export time'),
            ])
            list_files.append(df)
        except Exception as e:
            print(f"❌ Error reading {filename.name}: {e}")
            continue
    if list_files:
        return pl.concat(list_files, how='diagonal_relaxed')
    return pl.DataFrame()


outage_db = input_data(DATA_DIR)
if outage_db.is_empty():
    raise RuntimeError(f"❌ Không load được file nào từ:\n{DATA_DIR}")
print(f"✅ Loaded {len(outage_db)} rows")

outage_db = outage_db.sort(["Export time"])
outage_db = outage_db.filter(pl.col("Export time") == pl.col("Export time").max())

# ── LOB Mapping — chỉ NL Chat + LG Chat ─────────────────────────
LOB_MAP = {
    "LG Chat": [
        "Chat_OD_EN_Car_Activity", "Chat_OD_EN_Lodging",
        "Chat - Global English Lodging Nesting", "Chat_Lodging English w Car",
        "Chat_AC_GLB_EN_Lodging_Proficient", "Chat_AC_GLB_EN_Car_Activity",
        "Chat_AC_GLB_EN_Lodging_Expert",
    ],
    "NL Chat": [
        "Chat - Global English Non- Lodging Nesting", "Chat_OD_EN_Dual_GDS",
        "Chat_AC_GLB_EN_Proficient", "Chat_AC_GLB_EN_Expert",
    ],
}

lob_expr = pl.lit(None).cast(pl.Utf8)
for lob_label, queues in LOB_MAP.items():
    lob_expr = (
        pl.when(pl.col("Queue Group / Routing Profile").is_in(queues))
        .then(pl.lit(lob_label))
        .otherwise(lob_expr)
    )
outage_db = outage_db.with_columns(lob_expr.alias("LOB"))

# ── Chỉ giữ NL Chat + LG Chat ────────────────────────────────────
outage_db = outage_db.filter(pl.col("LOB").is_in(["NL Chat", "LG Chat"]))
print(f"✅ After LOB filter: {len(outage_db)} rows | LOBs: {outage_db['LOB'].unique().to_list()}")

# ── Location mapping — ALL SITES ─────────────────────────────────
outage_db = outage_db.with_columns(
    pl.when(pl.col("Business Location").str.contains("Ho Chi Minh")).then(pl.lit("HCM"))
    .when(pl.col("Business Location").str.contains("Pune")).then(pl.lit("PUN"))
    .when(pl.col("Business Location").str.contains("Kolkata")).then(pl.lit("KOL"))
    .when(pl.col("Business Location").str.contains("Cairo")).then(pl.lit("CAI"))
    .otherwise(pl.lit("OTHER"))
    .alias("Location")
)
print(f"✅ Locations: {outage_db['Location'].unique().to_list()}")

# ── Duration → Seconds ───────────────────────────────────────────
def str_hms_to_seconds(hms):
    try:
        parts = [int(p) for p in str(hms).split(':')]
        if len(parts) == 3: return parts[0] * 3600 + parts[1] * 60 + parts[2]
        if len(parts) == 2: return parts[0] * 60 + parts[1]
        return int(parts[0])
    except Exception:
        return None

outage_db = outage_db.with_columns(
    pl.col("Duration").cast(str)
      .map_elements(str_hms_to_seconds, return_dtype=pl.Int64)
      .alias("Duration (s)")
)

# ════════════════════════════════════════════════════════════════
# Detail tables — ALL SITES, không filter site
# ════════════════════════════════════════════════════════════════

# 1a. Lunch / Break — tất cả agents đang break/lunch
break_lunch_outage = outage_db.filter(
    pl.col("Connect State").is_in(["BREAK", "LUNCH"])
).with_columns(
    pl.when(
        (pl.col("Connect State") == "BREAK") & (pl.col("Duration (s)") > 15 * 60)
    ).then(pl.lit("⚠️ over-break"))
    .when(
        (pl.col("Connect State") == "LUNCH") & (pl.col("Duration (s)") > 60 * 60)
    ).then(pl.lit("⚠️ over-lunch"))
    .otherwise(pl.lit("OK"))
    .alias("Note")
)

# 1b. Overbreak / Overlunch — chỉ các agents đã vượt threshold
over_outage = break_lunch_outage.filter(pl.col("Note") != "OK")

# Summary count per LOB cho subtitle
bl_counts = (
    break_lunch_outage
    .group_by(["LOB", "Connect State"])
    .agg(pl.len().alias("Count"))
    .sort(["LOB", "Connect State"])
)
bl_parts: dict = {}
for row in bl_counts.iter_rows(named=True):
    bl_parts.setdefault(row["LOB"], []).append(f"{row['Connect State']} ×{row['Count']}")
bl_summary_str = "  |  ".join(
    f"<b>{lob}</b>: {', '.join(parts)}"
    for lob, parts in sorted(bl_parts.items())
)

over_counts = (
    over_outage
    .group_by(["LOB", "Connect State"])
    .agg(pl.len().alias("Count"))
    .sort(["LOB", "Connect State"])
)
over_parts: dict = {}
for row in over_counts.iter_rows(named=True):
    over_parts.setdefault(row["LOB"], []).append(f"{row['Connect State']} ×{row['Count']}")
over_summary_str = "  |  ".join(
    f"<b>{lob}</b>: {', '.join(parts)}"
    for lob, parts in sorted(over_parts.items())
) if over_parts else "No violations"

# 2. Coaching / Training / Unproductive — tất cả state không thuộc nhóm khác
coaching_training_outage = outage_db.filter(
    ~pl.col("Connect State").is_in(["BREAK", "LUNCH", "AVAILABLE", "READY", "OFFLINEWORK"])
).with_columns(
    pl.when(pl.col("Connect State").is_in(["COACHING", "TRAINING", "TEAM MEETING"]))
      .then(pl.lit("🔍 need to check"))
    .otherwise(pl.lit("⚠️ unproductive"))
    .alias("Note")
)

# 3. Available-Idle / Offline-with-work
active_outage = outage_db.filter(
    pl.col("Connect State").is_in(["AVAILABLE", "READY", "OFFLINEWORK"])
).with_columns(
    pl.when(
        pl.col("Connect State").is_in(["AVAILABLE", "READY"])
        & (pl.col("Assigned Workitem Count").is_null() | (pl.col("Assigned Workitem Count") == 0))
        & (pl.col("Duration (s)") > 30 * 60)
    ).then(pl.lit("⚠️ available idle"))
    .when(
        (pl.col("Connect State") == "OFFLINEWORK")
        & (pl.col("Assigned Workitem Count") >= 1)
        & (pl.col("Duration (s)") > 10 * 60)
    ).then(pl.lit("⚠️ offline w/ work"))
    .otherwise(pl.lit("OK"))
    .alias("Note")
)

# ── IC Overall Pivot — All Sites ──────────────────────────────────
all_categorized = outage_db.with_columns(
    pl.when(
        (pl.col("Assigned Workitem Count") >= 1)
        | pl.col("Connect State").is_in(["AVAILABLE", "READY"])
    ).then(pl.lit("Available"))
    .when(pl.col("Connect State") == "BREAK").then(pl.lit("Break-Idle"))
    .when(pl.col("Connect State") == "LUNCH").then(pl.lit("Lunch-Idle"))
    .when(pl.col("Connect State").is_in(["COACHING", "TEAM MEETING"])).then(pl.lit("Coaching-Idle"))
    .when(pl.col("Connect State") == "TRAINING").then(pl.lit("Training-Idle"))
    .otherwise(pl.lit("Other"))
    .alias("Category")
)

grouped = (
    all_categorized
    .group_by(["Location", "LOB", "Category"])
    .agg(pl.col("Agent Name").n_unique().alias("Count"))
)

pivot_global = (
    grouped.pivot(values="Count", index=["Location", "LOB"], columns="Category")
    .fill_null(0)
)

for col in ["Available", "Break-Idle", "Lunch-Idle", "Coaching-Idle", "Training-Idle", "Other"]:
    if col not in pivot_global.columns:
        pivot_global = pivot_global.with_columns(pl.lit(0).cast(pl.Int64).alias(col))

pivot_global = pivot_global.select(
    ["Location", "LOB", "Available", "Break-Idle", "Lunch-Idle",
     "Coaching-Idle", "Training-Idle", "Other"]
).sort(["Location", "LOB"])

# ── process_outage: sort LOB → State → Duration desc ─────────────
def process_outage(df: pl.DataFrame, select_cols: list) -> tuple:
    df = df.sort(
        ["LOB", "Connect State", "Duration (s)"],
        descending=[False, False, True]
    )
    total_cases = df.shape[0]
    df = df.head(50)
    available_cols = [c for c in select_cols if c in df.columns]
    return df.select(available_cols).to_pandas(), total_cases


BASE_COLS = ["Location", "Agent Name", "Agent Manager",
             "Connect State", "Duration (s)", "LOB", "Note"]

break_lunch_pd,       break_lunch_cases       = process_outage(break_lunch_outage, BASE_COLS)
over_pd,              over_cases              = process_outage(over_outage,         BASE_COLS)
coaching_training_pd, coaching_training_cases = process_outage(coaching_training_outage, BASE_COLS)
active_pd,            active_cases            = process_outage(
    active_outage,
    ["Location", "Agent Name", "Agent Manager",
     "Connect State", "Assigned Workitem Count", "Duration (s)", "LOB", "Note"]
)
pivot_pd = pivot_global.to_pandas()

print(f"\n{'─'*55}")
print(f"IC Overall          : {len(pivot_pd)} rows")
print(f"Lunch/Break         : {break_lunch_cases} cases  →  {bl_summary_str}")
print(f"Overbreak/Overlunch : {over_cases} cases  →  {over_summary_str}")
print(f"Coaching/Training   : {coaching_training_cases} cases")
print(f"Available-Idle      : {active_cases} cases")
print(f"{'─'*55}")
print("👆 Kiểm tra xong → chạy Cell gửi webhook")

✅ Loaded 5468 rows
✅ After LOB filter: 79 rows | LOBs: ['NL Chat', 'LG Chat']
✅ Locations: ['HCM', 'CAI', 'KOL']

───────────────────────────────────────────────────────
IC Overall          : 5 rows
Lunch/Break         : 15 cases  →  <b>LG Chat</b>: BREAK ×1, LUNCH ×7  |  <b>NL Chat</b>: BREAK ×2, LUNCH ×5
Overbreak/Overlunch : 0 cases  →  No violations
Coaching/Training   : 4 cases
Available-Idle      : 60 cases
───────────────────────────────────────────────────────
👆 Kiểm tra xong → chạy Cell gửi webhook


C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_21944\3095615178.py:209: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  grouped.pivot(values="Count", index=["Location", "LOB"], columns="Category")


In [65]:
import re
from datetime import time as dtime

SCHEDULE_FILE = (
    r"C:\Users\huuchinh.nguyen\Concentrix Corporation"
    r"\WFM-Expedia-HCM - Branding files\Schedule\Schedule (Ops version)"
    r"\2026\Master_Schedule_Merged.xlsx"
)

LEAVE_CODES    = {'AL', 'LWP', 'CO', 'SL', 'EL', 'ML', 'PL', 'SPL', 'BL', 'CL'}
EXCLUDE_SHIFTS = {'OFF', 'TERMINATION', 'TERM', 'RESIGNED'}

now_dt    = datetime.now()
if now_dt.hour < 5:
    today = now_dt.date() - timedelta(days=1)
else:
    today = now_dt.date()
yesterday = today - timedelta(days=1)

def input_data_all_raw(data_dir):
    list_files = []
    for filename in pathlib.Path(data_dir).glob('**/*.*'):
        file_suffixes = filename.suffixes
        if not (file_suffixes and file_suffixes[-1].lower() in ['.xlsx', '.csv']):
            continue
        try:
            export_time          = os.path.getmtime(filename)
            export_time_datetime = convert_to_datetime(time.localtime(export_time))
            if file_suffixes[-1].lower() == '.xlsx':
                df = pl.read_excel(filename)
            else:
                if os.path.getsize(filename) == 0:
                    continue
                df = pl.read_csv(filename, infer_schema_length=10000)
            if df.is_empty():
                continue
            df = df.with_columns(
                pl.lit(export_time_datetime).alias('Export time')
            )
            list_files.append(df)
        except Exception:
            continue
    if list_files:
        return pl.concat(list_files, how='diagonal_relaxed')
    return pl.DataFrame()

raw_all = input_data_all_raw(DATA_DIR)
print(f"✅ All snapshots combined: {len(raw_all)} rows")

lob_expr_all = pl.lit(None).cast(pl.Utf8)
for lob_label, queues in LOB_MAP.items():
    lob_expr_all = (
        pl.when(pl.col("Queue Group / Routing Profile").is_in(queues))
        .then(pl.lit(lob_label))
        .otherwise(lob_expr_all)
    )

raw_hcm = (
    raw_all
    .with_columns(lob_expr_all.alias("LOB"))
    .filter(pl.col("LOB").is_in(["NL Chat", "LG Chat"]))
    .with_columns(
        pl.when(pl.col("Business Location").str.contains("Ho Chi Minh")).then(pl.lit("HCM"))
        .when(pl.col("Business Location").str.contains("Pune")).then(pl.lit("PUN"))
        .when(pl.col("Business Location").str.contains("Kolkata")).then(pl.lit("KOL"))
        .when(pl.col("Business Location").str.contains("Cairo")).then(pl.lit("CAI"))
        .otherwise(pl.lit("OTHER"))
        .alias("Location")
    )
    .filter(pl.col("Location") == "HCM")
)

email_lob_map = (
    raw_hcm.select(["Agent Email", "LOB"]).drop_nulls()
    .unique(subset=["Agent Email"], keep="last")
    .to_pandas()
    .assign(Email_key=lambda df: df["Agent Email"].str.strip().str.lower())
    .set_index("Email_key")["LOB"].to_dict()
)
print(f"✅ Email→LOB map: {len(email_lob_map)} agents")

hcm_email_keys = set(email_lob_map.keys())

login_raw = (
    raw_hcm
    .select(["Agent Name", "Agent Email", "Login Time"])
    .to_pandas()
    .dropna(subset=["Agent Email", "Login Time"])
)
login_raw["Login_DT"]   = pd.to_datetime(login_raw["Login Time"], errors="coerce")
login_raw["Login_Date"] = login_raw["Login_DT"].dt.date
login_raw["Email_key"]  = login_raw["Agent Email"].str.strip().str.lower()

login_today = (
    login_raw[login_raw["Login_Date"] == today]
    .dropna(subset=["Login_DT"])
    .sort_values("Login_DT")
    .drop_duplicates("Email_key", keep="first")
    [["Email_key", "Agent Name", "Agent Email", "Login_DT"]]
    .reset_index(drop=True)
)
print(f"✅ Unique HCM logins today: {len(login_today)}")
state_latest = (
    raw_hcm
    .filter(pl.col("Export time") == pl.col("Export time").max())
    .select(["Agent Email", "Connect State"])
    .to_pandas()
    .assign(Email_key=lambda df: df["Agent Email"].str.strip().str.lower())
    .drop_duplicates("Email_key", keep="last")
    [["Email_key", "Connect State"]]
    .reset_index(drop=True)
)
print(f"✅ Latest snapshot agents: {len(state_latest)} | Export time: {raw_hcm.select('Export time').max().item()}")
sched_raw = pd.read_excel(SCHEDULE_FILE, dtype=str)
sched_raw.columns = [str(c).strip() for c in sched_raw.columns]

date_cols_raw = [c for c in sched_raw.columns if re.match(r'^\d{4}-\d{2}-\d{2}', c)]
col_map       = {c: c[:10] for c in date_cols_raw}
sched_raw     = sched_raw.rename(columns=col_map)
date_cols     = list(col_map.values())
keep_dates    = [str(d) for d in [yesterday, today] if str(d) in date_cols]
print(f"✅ Schedule dates found: {keep_dates}")

email_col = "Email"
name_col  = "Employee Name" if "Employee Name" in sched_raw.columns else None
id_cols   = [email_col] + ([name_col] if name_col else [])

sched_long = (
    sched_raw[id_cols + keep_dates]
    .melt(id_vars=id_cols, value_vars=keep_dates, var_name="Sched_Date", value_name="Shift")
    .assign(
        Sched_Date = lambda df: pd.to_datetime(df["Sched_Date"]).dt.date,
        Email_key  = lambda df: df[email_col].astype(str).str.strip().str.lower(),
        Shift      = lambda df: df["Shift"].astype(str).str.strip()
    )
    .loc[lambda df: ~df["Shift"].isin(["", "nan", "None"])]
    .drop_duplicates(["Email_key", "Sched_Date"])
    .reset_index(drop=True)
)

sched_long = sched_long[sched_long["Email_key"].isin(hcm_email_keys)].reset_index(drop=True)
print(f"✅ Schedule after HCM filter: {len(sched_long)} rows")


def parse_shift_start(shift_str: str):
    try:
        start = str(shift_str).strip().split("-")[0]
        if len(start) == 4 and start.isdigit():
            return dtime(int(start[:2]), int(start[2:]))
    except Exception:
        pass
    return None

def is_overnight(shift_str: str) -> bool:
    try:
        parts = str(shift_str).strip().split("-")
        if len(parts) == 2 and all(len(p) == 4 and p.isdigit() for p in parts):
            return int(parts[1][:2]) <= 8
    except Exception:
        pass
    return False

sched_long["Shift_Start_DT"] = sched_long.apply(
    lambda r: (
        datetime.combine(r["Sched_Date"], parse_shift_start(r["Shift"]))
        if "-" in str(r["Shift"]) and parse_shift_start(r["Shift"])
        else None
    ), axis=1
)

def should_include(row) -> bool:
    shift = str(row["Shift"]).strip()
    upper = shift.upper()
    if upper in EXCLUDE_SHIFTS or "TERMINAT" in upper:
        return False
    if upper in LEAVE_CODES or "-" not in shift:
        return row["Sched_Date"] == today
    start_dt = row["Shift_Start_DT"]
    if start_dt is None:
        return False
    if row["Sched_Date"] == today:
        return start_dt <= now_dt
    elif row["Sched_Date"] == yesterday:
        if not is_overnight(shift):
            return False
        try:
            end_str = shift.strip().split("-")[1]
            end_dt  = datetime.combine(today, dtime(int(end_str[:2]), int(end_str[2:])))
            return end_dt >= now_dt
        except Exception:
            return False
    return False

sched_active = (
    sched_long[sched_long.apply(should_include, axis=1)]
    .copy()
    .assign(_sort_key=lambda df: df["Shift_Start_DT"].fillna(pd.Timestamp("2099-01-01")))
    .sort_values("_sort_key")
    .reset_index(drop=True)
)
print(f"✅ Active shifts as of {now_dt.strftime('%H:%M')}: {len(sched_active)}")

# Merge 1: schedule + earliest Login Time
merged = sched_active.merge(login_today, on="Email_key", how="left")

# Merge 2: + latest Connect State
merged = merged.merge(state_latest, on="Email_key", how="left")

if name_col:
    merged["Agent Name"] = merged["Agent Name"].fillna(merged[name_col])
merged["Agent Email"] = merged["Agent Email"].fillna(merged[email_col])
merged["LOB"]         = merged["Email_key"].map(email_lob_map).fillna("—")

merged["Connect State"] = merged.apply(
    lambda r: "DROPPED"
    if (pd.notna(r.get("Login_DT")) and
        (pd.isna(r.get("Connect State")) or str(r.get("Connect State")).strip() in ["", "nan", "None"]))
    else (r.get("Connect State") if pd.notna(r.get("Connect State")) else "—"),
    axis=1
)



def calc_attendance(row):
    shift = str(row["Shift"]).strip()
    upper = shift.upper()
    if upper in LEAVE_CODES or "-" not in shift:
        return upper, None
    login_dt = row.get("Login_DT")
    start_dt = row["Shift_Start_DT"]
    if pd.isna(login_dt) or start_dt is None:
        return "ABS", None
    diff = (login_dt - start_dt).total_seconds()
    if diff < -12 * 3600: diff += 86400
    elif diff > 12 * 3600: diff -= 86400
    if diff > 60:
        h, rem = divmod(int(diff), 3600)
        m, s   = divmod(rem, 60)
        return "PR", f"{h:02d}:{m:02d}:{s:02d}"
    return "PR", None

res = merged.apply(calc_attendance, axis=1, result_type="expand")
merged["ATD Code"]   = res[0]
merged["Late"]       = res[1]
merged["Login Time"] = merged["Login_DT"].apply(
    lambda x: x.strftime("%H:%M:%S") if pd.notna(x) else "—"
)

attendance_pd = (
    merged[["Agent Name", "Agent Email", "Login Time",
            "Shift", "LOB", "Connect State", "Late", "ATD Code"]]
    .reset_index(drop=True)
)

_lob_str = attendance_pd["LOB"].astype(str).str.strip()
attendance_pd = attendance_pd[
    _lob_str.notna() &
    (_lob_str != "") &
    (_lob_str != "nan") &
    (_lob_str != "None") &
    (_lob_str != "—") &
    (_lob_str != "-")
].reset_index(drop=True)

attendance_cases = len(attendance_pd)
pr_count    = (attendance_pd["ATD Code"] == "PR").sum()
abs_count   = (attendance_pd["ATD Code"] == "ABS").sum()
late_count  = attendance_pd["Late"].notna().sum()
leave_count = (~attendance_pd["ATD Code"].isin(["PR", "ABS"])).sum()

print(f"\n{'─'*55}")
print(f"Attendance (VN)   : {attendance_cases} agents")
print(f"  ✅ PR           : {pr_count}  |  ⏰ Late: {late_count}")
print(f"  ❌ ABS          : {abs_count}  |  📅 Leave: {leave_count}")
print(f"{'─'*55}")
print(attendance_pd.to_string())

✅ All snapshots combined: 5468 rows
✅ Email→LOB map: 67 agents
✅ Unique HCM logins today: 29
✅ Latest snapshot agents: 28 | Export time: 2026-07-30 09:22:13
✅ Schedule dates found: ['2026-07-29', '2026-07-30']
✅ Schedule after HCM filter: 134 rows
✅ Active shifts as of 09:22: 33

───────────────────────────────────────────────────────
Attendance (VN)   : 33 agents
  ✅ PR           : 29  |  ⏰ Late: 19
  ❌ ABS          : 1  |  📅 Leave: 3
───────────────────────────────────────────────────────
                 Agent Name                            Agent Email Login Time      Shift      LOB Connect State      Late ATD Code
0              Duy Tinh Ngu             duytinh.ngu@concentrix.com   05:00:39  0500-1400  LG Chat     AVAILABLE      None       PR
1               Tien Dat Vo              tiendat.vo@concentrix.com   04:58:11  0500-1400  LG Chat         LUNCH      None       PR
2       Nguyen Nhat Anh Huy       nguyennhatanh.huy@concentrix.com   05:19:06  0500-1400  NL Chat     AVAILABLE

In [66]:
# ════════════════════════════════════════════════════════════════
# CELL 5 ▸ Preview Tables
# ════════════════════════════════════════════════════════════════
from IPython.display import display

for name, df in [
    ("IC Overall",          pivot_pd),
    ("Break/Lunch Outage",  break_lunch_pd),
    ("Coaching/Training",   coaching_training_pd),
    ("Available-Idle",      active_pd),
]:
    print(f"\n{'\u2500'*50}\n\u2705  {name}  ({len(df)} rows)")
    display(df)



──────────────────────────────────────────────────
✅  IC Overall  (5 rows)


,Location,LOB,Available,Break-Idle,Lunch-Idle,Coaching-Idle,Training-Idle,Other
0,CAI,NL Chat,3,0,1,0,0,0
1,HCM,LG Chat,16,1,6,0,0,0
2,HCM,NL Chat,3,0,2,0,0,0
3,KOL,LG Chat,18,0,1,2,0,0
4,KOL,NL Chat,20,2,2,0,0,2



──────────────────────────────────────────────────
✅  Break/Lunch Outage  (15 rows)


,Location,Agent Name,Agent Manager,Connect State,Duration (s),LOB,Note
0,HCM,Thuy Tram Nguyen,thienthanhtoan.truong@concen,BREAK,750,LG Chat,OK
1,HCM,Cong Danh Bui,thaouyen.tran1@concentrix.co,LUNCH,2528,LG Chat,OK
2,HCM,Thi Diem Quynh Nguyen,hoangmyanh.tran@concentrix.c,LUNCH,1620,LG Chat,OK
3,HCM,Ngoc Tram Phung,thaouyen.tran1@concentrix.co,LUNCH,1180,LG Chat,OK
4,HCM,Thi Tuyet Mai Nguyen,thaouyen.tran1@concentrix.co,LUNCH,1173,LG Chat,OK
5,HCM,Ngoc Hien Ly,thienthanhtoan.truong@concen,LUNCH,1052,LG Chat,OK
6,KOL,RIYA KUMARI,amit.lamba@concentrix.com,LUNCH,563,LG Chat,OK
7,HCM,Tien Dat Vo,thaouyen.tran1@concentrix.co,LUNCH,522,LG Chat,OK
8,KOL,nigel rozario,None,BREAK,769,NL Chat,OK
9,KOL,Pritam Majumdar,None,BREAK,336,NL Chat,OK



──────────────────────────────────────────────────
✅  Coaching/Training  (4 rows)


,Location,Agent Name,Agent Manager,Connect State,Duration (s),LOB,Note
0,KOL,SWASTIKA GHOSH,amit.lamba@concentrix.com,COACHING,562,LG Chat,🔍 need to check
1,KOL,CHARU SHARMA,amit.lamba@concentrix.com,COACHING,224,LG Chat,🔍 need to check
2,KOL,Gurvir Singh,amit.lamba@concentrix.com,LOGIN,9982,NL Chat,⚠️ unproductive
3,KOL,Shalini Roy,None,PERSONAL,438,NL Chat,⚠️ unproductive



──────────────────────────────────────────────────
✅  Available-Idle  (50 rows)


,Location,Agent Name,Agent Manager,Connect State,Assigned Workitem Count,Duration (s),LOB,Note
0,KOL,Atreyee Gayen,amit.lamba@concentrix.com,AVAILABLE,1.0,1586,LG Chat,OK
1,KOL,Nazish Sawa,amit.lamba@concentrix.com,AVAILABLE,1.0,1216,LG Chat,OK
2,KOL,RAKHEE SINGH,None,AVAILABLE,1.0,725,LG Chat,OK
3,KOL,NIDHI KUMARI SHAW,amit.lamba@concentrix.com,AVAILABLE,NaN,508,LG Chat,OK
4,HCM,Ngoc Thuan Vy Bui,thienthanhtoan.truong@concen,AVAILABLE,1.0,462,LG Chat,OK
5,HCM,Thi Diem Quynh Dang,thienthanhtoan.truong@concen,AVAILABLE,1.0,453,LG Chat,OK
6,HCM,Chau Bao Yen Tran,thienthanhtoan.truong@concen,AVAILABLE,1.0,389,LG Chat,OK
7,HCM,Van Phuc Bui,thienthanhtoan.truong@concen,AVAILABLE,1.0,375,LG Chat,OK
8,HCM,Tuan Anh Nguyen,thienthanhtoan.truong@concen,AVAILABLE,1.0,298,LG Chat,OK
9,KOL,ASHISH KUMAR PRASAD,None,AVAILABLE,2.0,258,LG Chat,OK


In [67]:
# ════════════════════════════════════════════════════════════════
# CELL 6 ▸ Build HTML Tables + Send via Webhook
# ════════════════════════════════════════════════════════════════

# ── Style maps ────────────────────────────────────────────────

LOC_STYLE = {
    "HCM":   {"bg": "#DA251D", "fg": "#FFD700"},
    "PUN":   {"bg": "#388E3C", "fg": "#ffffff"},
    "KOL":   {"bg": "#1565C0", "fg": "#ffffff"},
    "CAI":   {"bg": "#E65100", "fg": "#ffffff"},
    "OTHER": {"bg": "#757575", "fg": "#ffffff"},
}

LOB_STYLE = {
    "LG Chat":  {"bg": "#1565C0", "fg": "#ffffff"},
    "NL Chat":  {"bg": "#2E7D32", "fg": "#ffffff"},
    "LG Voice": {"bg": "#4A148C", "fg": "#ffffff"},
    "NL Voice": {"bg": "#BF360C", "fg": "#ffffff"},
}

NOTE_STYLE = {
    "⚠️ over-break":      {"bg": "#E65100", "fg": "#ffffff"},
    "⚠️ over-lunch":      {"bg": "#B71C1C", "fg": "#ffffff"},
    "⚠️ available idle":  {"bg": "#F57F17", "fg": "#ffffff"},
    "⚠️ offline w/ work": {"bg": "#880E4F", "fg": "#ffffff"},
    "🔍 need to check":   {"bg": "#4A148C", "fg": "#ffffff"},
    "OK":                         {"bg": "#E8F5E9", "fg": "#2E7D32"},
}

CONNECT_STATE_STYLE = {
    "AVAILABLE":    ("#E3F2FD", "#1565C0"),
    "READY":        ("#E3F2FD", "#1565C0"),
    "BREAK":        ("#FFE0B2", "#BF360C"), 
    "LUNCH":        ("#C8E6C9", "#1B5E20"), 
    "COACHING":     ("#EDE7F6", "#4A148C"),
    "TRAINING":     ("#E8EAF6", "#283593"),
    "TEAM MEETING": ("#E8EAF6", "#283593"),
    "OFFLINEWORK":  ("#FFCCBC", "#BF360C"),
    "NOT READY":    ("#FFEBEE", "#C62828"),
    "NOTREADY":     ("#FFEBEE", "#C62828"),
    "UNAVAILABLE":  ("#FFEBEE", "#C62828"),
    "ENDOFSHIFT":   ("#FAFAFA", "#757575"),
    "LOGIN":        ("#FFF8E1", "#F57F17"),
    "PERSONAL":     ("#FBE9E7", "#BF360C"),
    "DROPPED":      ("#B71C1C", "#ffffff"),
}

ATD_CODE_STYLE = {
    'PR':  {'bg': '#1B5E20', 'fg': '#ffffff'},   # xanh đậm
    'ABS': {'bg': '#B71C1C', 'fg': '#ffffff'},   # đỏ đậm
    'AL':  {'bg': '#1A237E', 'fg': '#ffffff'},   # xanh navy
    'LWP': {'bg': '#4A148C', 'fg': '#ffffff'},   # tím
    'CO':  {'bg': '#006064', 'fg': '#ffffff'},   # xanh dầu
    'SL':  {'bg': '#37474F', 'fg': '#ffffff'},   # xám
    'EL':  {'bg': '#37474F', 'fg': '#ffffff'},
    'ML':  {'bg': '#37474F', 'fg': '#ffffff'},
}

def get_shift_style(shift_val: str):
    """Morning 05-10 → green | Mid 11-17 → orange | Night 18-04 → blue"""
    t = parse_shift_start(str(shift_val))
    if t is None:
        return None, None
    h = t.hour
    if 5 <= h <= 10:
        return "#2E7D32", "#ffffff"   # morning — xanh lá
    elif 11 <= h <= 17:
        return "#E65100", "#ffffff"   # mid     — cam
    else:
        return "#1565C0", "#ffffff"   # night   — xanh biển
    
# ── Helpers ───────────────────────────────────────────────
def seconds_to_hms(s) -> str:
    try:
        s = int(float(s))
        h, r   = divmod(s, 3600)
        m, sec = divmod(r, 60)
        return f'{h:02d}:{m:02d}:{sec:02d}'
    except Exception:
        return '—'


def hms_to_seconds(hms) -> int:
    try:
        p = str(hms).split(':')
        return int(p[0]) * 3600 + int(p[1]) * 60 + int(p[2])
    except Exception:
        return 0


def get_duration_color(hms_str: str):
    """Color by absolute duration bands."""
    try:
        s = hms_to_seconds(hms_str)
        if s > 3600: return "#b71c1c", "#ffffff"   # > 1 hr
        if s > 1800: return "#e53935", "#ffffff"   # > 30 min
        if s >  900: return "#fb8c00", "#ffffff"   # > 15 min
        if s >  300: return "#fdd835", "#1a1a1a"  # >  5 min
    except Exception:
        pass
    return None, None


def make_bar_cell(s: int, max_s: int, color):
    ratio  = s / max_s if max_s > 0 else 0
    filled = int(ratio * 150)
    empty  = 150 - filled
    bar_bg = color if color else "#43a047"
    nb_fill = '&nbsp;' if filled > 0 else ''
    nb_empty = '&nbsp;' if empty > 0 else ''
    return (
        '<td style="padding:3px 8px;border:1px solid #ddd;vertical-align:middle;" nowrap>'
        '<table cellpadding="0" cellspacing="0" border="0" width="150" style="border-collapse:collapse;">'
        '<tr height="14">'
        f'<td width="{filled}" height="14" bgcolor="{bar_bg}" '
        f'style="height:14px;font-size:10px;line-height:14px;">{nb_fill}</td>'
        f'<td width="{empty}" height="14" bgcolor="#e0e0e0" '
        f'style="height:14px;font-size:10px;line-height:14px;">{nb_empty}</td>'
        '</tr></table></td>'
    )


# ── HTML Table Builder (follows lc_2_capture pattern) ────────────
def build_html_table(df: pd.DataFrame, title: str,
                     is_global: bool = False, cases: int = 0,
                     summary: str = '') -> str:
    now      = datetime.now() - (timedelta(hours=14) if is_global else timedelta(0))
    timezone = '(PST)' if is_global else '(VNT)'
    subtitle = f"Updated {now.strftime('%d-%b-%Y')} · {now.strftime('%I:%M %p')} {timezone}"

    df = df.copy()
    df = df.drop(columns=[c for c in ['sheet_name', 'Export time'] if c in df.columns])

    if 'Duration (s)' in df.columns:
        df['Duration'] = df['Duration (s)'].apply(seconds_to_hms)
        df = df.drop(columns=['Duration (s)'])

    col_names  = list(df.columns)
    dur_idx    = col_names.index('Duration')      if 'Duration'      in col_names else -1
    lob_idx    = col_names.index('LOB')           if 'LOB'           in col_names else -1
    note_idx   = col_names.index('Note')          if 'Note'          in col_names else -1
    state_idx  = col_names.index('Connect State') if 'Connect State' in col_names else -1
    atd_idx  = col_names.index('ATD Code') if 'ATD Code' in col_names else -1
    late_idx = col_names.index('Late')     if 'Late'     in col_names else -1
    shift_idx = col_names.index("Shift") if "Shift" in col_names else -1
    loc_idx    = col_names.index('Location')      if 'Location'      in col_names else -1

    if dur_idx >= 0:
        dur_secs = [hms_to_seconds(str(row[col_names[dur_idx]])) for _, row in df.iterrows()]
        max_s    = max(dur_secs) if dur_secs else 1
    else:
        dur_secs = [0] * len(df)
        max_s    = 1

    th = 'bgcolor="#1e3a5f" style="color:#ffffff;padding:7px 12px;border:1px solid #2c4f7c;text-align:left;" nowrap'
    header_list = [f'<th {th}>{c}</th>' for c in col_names]
    if dur_idx >= 0:
        header_list.insert(dur_idx + 1, f'<th {th}>Duration Bar</th>')
    headers = ''.join(header_list)

    def td_plain(bg, val):
        return (
            f'<td bgcolor="{bg}" style="color:#1a1a1a;padding:6px 12px;'
            f'border:1px solid #ddd;" nowrap>{val}</td>'
        )

    def td_bold(bg, fg, val, align='left'):
        return (
            f'<td bgcolor="{bg}" style="color:{fg};padding:6px 12px;'
            f'border:1px solid #ddd;font-weight:bold;text-align:{align};" nowrap>{val}</td>'
        )

    rows_html = ''
    for i, (_, row) in enumerate(df.iterrows()):
        row_bg   = '#f0f4ff' if i % 2 == 0 else '#ffffff'
        lob_val  = str(row[col_names[lob_idx]])  if lob_idx  >= 0 else ''
        note_val = str(row[col_names[note_idx]]) if note_idx >= 0 else ''
        cells    = []

        for j, col in enumerate(col_names):
            val = str(row[col]) if pd.notna(row[col]) else '—'

            if j == dur_idx:
                bg, fg = get_duration_color(val)
                if bg:
                    cells.append(td_bold(bg, fg, val))
                else:
                    cells.append(td_plain(row_bg, val))
                cells.append(make_bar_cell(dur_secs[i], max_s, bg))

            elif j == loc_idx:
                loc_s = LOC_STYLE.get(val)
                if loc_s:
                    cells.append(td_bold(loc_s['bg'], loc_s['fg'], val, align='center'))
                else:
                    cells.append(td_plain(row_bg, val))

            elif j == lob_idx:
                lob_s = LOB_STYLE.get(lob_val)
                if lob_s:
                    cells.append(td_bold(lob_s['bg'], lob_s['fg'], val, align='center'))
                else:
                    cells.append(td_plain(row_bg, val))

            elif j == note_idx:
                note_s = NOTE_STYLE.get(note_val, {'bg': row_bg, 'fg': '#1a1a1a'})
                cells.append(td_bold(note_s['bg'], note_s['fg'], val, align='center'))

            elif j == state_idx:
                sc = CONNECT_STATE_STYLE.get(val)
                if sc:
                    sbg, sfg = sc
                    cells.append(td_bold(sbg, sfg, val))
                else:
                    cells.append(td_plain(row_bg, val))
            elif j == atd_idx:
                atd_s = ATD_CODE_STYLE.get(val, {'bg': row_bg, 'fg': '#1a1a1a'})
                cells.append(td_bold(atd_s['bg'], atd_s['fg'], val, align='center'))

            elif j == late_idx:
                if val and val != '—' and val != 'None':
                    cells.append(td_bold('#E65100', '#ffffff', val))
                else:
                    cells.append(td_plain(row_bg, '—'))
            elif j == shift_idx:
                sbg, sfg = get_shift_style(val)
                if sbg:
                    cells.append(td_bold(sbg, sfg, val, align="center"))
                else:
                    cells.append(td_plain(row_bg, val))
            else:
                cells.append(td_plain(row_bg, val))

        rows_html += f"<tr>{''.join(cells)}</tr>"

    summary_html = (
        f'  <span style="font-size:11px;">📊 {summary}</span><br>\n'
        if summary else ''
    )

    return (
        f'<p>\n'
        f'  <b style="color:#c0392b;font-size:16px;">🔴 {title}</b><br>\n'
        f'  <span style="font-size:12px;">{subtitle} &nbsp;|&nbsp; ⚡ <b>{cases} CASES</b></span><br>\n'
        f'{summary_html}'
        f'</p>\n'
        f'<div style="overflow-x:auto;">\n'
        f'<table border="1" cellpadding="0" cellspacing="0"\n'
        f'       style="border-collapse:collapse;font-size:12px;font-family:Segoe UI,Arial,sans-serif;">\n'
        f'  <thead><tr>{headers}</tr></thead>\n'
        f'  <tbody>{rows_html}</tbody>\n'
        f'</table>\n'
        f'</div>'
    )


# ── Send via Webhook (same pattern as lc_2_capture) ───────────────
def send_html_in_chunks(df, title, is_global=False, cases=0, summary="", chunk_size=25):
    """Chunk table để tránh RequestEntityTooLarge."""
    total    = len(df)
    n_chunks = max(1, (total + chunk_size - 1) // chunk_size)
    for i in range(n_chunks):
        chunk = df.iloc[i * chunk_size:(i + 1) * chunk_size].reset_index(drop=True)
        send_html_via_webhook(
            chunk,
            f"{title} ({i+1}/{n_chunks})" if n_chunks > 1 else title,
            is_global,
            cases if i == 0 else len(chunk),
            summary if i == 0 else "",
        )


def send_html_via_webhook(df, title, is_global=False, cases=0, summary=''):
    if df is None or (hasattr(df, 'empty') and df.empty):
        print(f"⏭️  Skipping '{title}' — empty")
        return
    payload = {'html': build_html_table(df, title, is_global, cases, summary)}
    try:
        resp = requests.post(
            TEAMS_WEBHOOK_URL,
            headers={'Content-Type': 'application/json'},
            data=json.dumps(payload),
            timeout=30,
        )
        if resp.status_code in (200, 202):
            print(f"✅ Sent: '{title}'  ({cases} cases)")
        else:
            print(f"❌ Failed [{resp.status_code}]: '{title}'\n   {resp.text[:300]}")
    except Exception as e:
        print(f"❌ Error: {e}")

send_html_via_webhook(pivot_pd,             'IC Overall — All Sites',             is_global=False, cases=len(pivot_pd))
send_html_via_webhook(break_lunch_pd,       'Lunch / Break',                      is_global=False, cases=break_lunch_cases,  summary=bl_summary_str)
send_html_via_webhook(over_pd,              'Overbreak / Overlunch',              is_global=False, cases=over_cases,          summary=over_summary_str)
send_html_via_webhook(coaching_training_pd, 'Coaching / Training / Unproductive', is_global=False, cases=coaching_training_cases)
send_html_via_webhook(active_pd,            'Available-IDLE',                     is_global=False, cases=active_cases)

attendance_pd["_shift_group"] = attendance_pd["Shift"].apply(
    lambda s: "Planned" if str(s).upper() in LEAVE_CODES else s
)

def _group_sort_key(g):
    if g == "Planned":
        return "9999"
    t = parse_shift_start(str(g))
    return f"{t.hour:02d}{t.minute:02d}" if t else "8888"

shift_order = sorted(
    attendance_pd["_shift_group"].unique().tolist(),
    key=_group_sort_key
)

for shift_val in shift_order:
    shift_df = (
        attendance_pd[attendance_pd["_shift_group"] == shift_val]
        .drop(columns=["_shift_group"])
        .reset_index(drop=True)
    )
    if shift_df.empty:
        continue

    pr_n   = (shift_df["ATD Code"] == "PR").sum()
    abs_n  = (shift_df["ATD Code"] == "ABS").sum()
    late_n = shift_df["Late"].notna().sum()
    summary_shift = f"PR: {pr_n} | ABS: {abs_n} | Late: {late_n}"

    label = "Planned" if shift_val == "Planned" else f"Shift {shift_val}"
    send_html_via_webhook(
        shift_df,
        f"Attendance — VN | {label}",
        is_global=False,
        cases=len(shift_df),
        summary=summary_shift,
    )

✅ Sent: 'IC Overall — All Sites'  (5 cases)
✅ Sent: 'Lunch / Break'  (15 cases)
⏭️  Skipping 'Overbreak / Overlunch' — empty
✅ Sent: 'Coaching / Training / Unproductive'  (4 cases)
✅ Sent: 'Available-IDLE'  (60 cases)
✅ Sent: 'Attendance — VN | Shift 0500-1400'  (14 cases)
✅ Sent: 'Attendance — VN | Shift 0600-1500'  (16 cases)
✅ Sent: 'Attendance — VN | Planned'  (3 cases)
